In [ ]:
import math
import os
import random
from collections import Counter, defaultdict

# ============================================================
# Assignment 6: Smoothing for Unigram, Bigram, Trigram,
# and Quadrigram language models
# ============================================================

CORPUS_FILE = os.path.join("..", "tokenized.txt")
NUM_SENTENCES = 1_000_000
DEV_SIZE = 1_000
TEST_SIZE = 1_000
SEED = 42
SOS, EOS, UNK = "<s>", "</s>", "<UNK>"
ORDERS = (1, 2, 3, 4)

# ------------------------------------------------------------
# 1. Load and split the corpus
# ------------------------------------------------------------
raw_sentences = []
with open(CORPUS_FILE, "r", encoding="utf-8") as corpus:
    for line in corpus:
        tokens = line.strip().split()
        if tokens:
            raw_sentences.append(tokens)
        if len(raw_sentences) >= NUM_SENTENCES:
            break

random.Random(SEED).shuffle(raw_sentences)
if len(raw_sentences) <= DEV_SIZE + TEST_SIZE:
    raise ValueError("The corpus must contain more than DEV_SIZE + TEST_SIZE sentences.")

split_at = len(raw_sentences) - DEV_SIZE - TEST_SIZE
train_sentences = raw_sentences[:split_at]
dev_sentences = raw_sentences[split_at:split_at + DEV_SIZE]
test_sentences = raw_sentences[split_at + DEV_SIZE:]

# Vocabulary is built only from training data. Unknown evaluation words become <UNK>.
vocab = {token for sentence in train_sentences for token in sentence}
vocab.update({SOS, EOS, UNK})
V = len(vocab)

def normalize_sentence(sentence):
    return [token if token in vocab else UNK for token in sentence]

def pad_sentence(sentence, n):
    return [SOS] * (n - 1) + normalize_sentence(sentence) + [EOS]

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

# ------------------------------------------------------------
# 2. Count n-grams and contexts
# ------------------------------------------------------------
counts = {n: Counter() for n in ORDERS}
contexts = {n: Counter() for n in (2, 3, 4)}
followers = {n: defaultdict(set) for n in (1, 2, 3, 4)}
predecessors = {n: defaultdict(set) for n in (1, 2, 3, 4)}

for sentence in train_sentences:
    for n in ORDERS:
        for gram in get_ngrams(pad_sentence(sentence, n), n):
            counts[n][gram] += 1
            if n > 1:
                contexts[n][gram[:-1]] += 1
            followers[n][gram[:-1]].add(gram[-1])
            predecessors[n][gram[-1]].add(gram[:-1])

N = sum(counts[1].values())

print(f"Corpus: {os.path.abspath(CORPUS_FILE)}")
print(f"Sentences: train={len(train_sentences):,}, dev={len(dev_sentences):,}, test={len(test_sentences):,}")
print(f"Vocabulary size: {V:,}")
for n in ORDERS:
    print(f"{n}-gram types: {len(counts[n]):,}; tokens: {sum(counts[n].values()):,}")

# ------------------------------------------------------------
# 3. Basic probability helpers
# ------------------------------------------------------------
def word_for_vocab(word):
    return word if word in vocab else UNK

def mle_prob(order, history, word):
    word = word_for_vocab(word)
    if order == 1:
        return counts[1].get((word,), 0) / N
    history = tuple(word_for_vocab(token) for token in history)
    denominator = contexts[order].get(history, 0)
    numerator = counts[order].get(history + (word,), 0)
    return numerator / denominator if denominator else 0.0

def safe_log_probability(probability):
    return math.log2(max(probability, 1e-300))

def sentence_log_probability(sentence, probability_function, order):
    padded = pad_sentence(sentence, order)
    total = 0.0
    token_count = len(padded) - order + 1
    for i in range(order - 1, len(padded)):
        history = tuple(padded[i - order + 1:i])
        total += safe_log_probability(probability_function(order, history, padded[i]))
    return total, token_count

# ------------------------------------------------------------
# 4. Interpolated smoothing
# ------------------------------------------------------------
# Weights are ordered from lower to higher model order and sum to 1.
INTERPOLATION_WEIGHTS = {
    1: (1.0,),
    2: (0.4, 0.6),
    3: (0.2, 0.3, 0.5),
    4: (0.1, 0.2, 0.3, 0.4),
}

def interpolated_prob(order, history, word):
    word = word_for_vocab(word)
    if order == 1:
        return max(mle_prob(1, (), word), 1 / N)
    weights = INTERPOLATION_WEIGHTS[order]
    probability = 0.0
    for lower_order, weight in enumerate(weights, start=1):
        lower_history = tuple(history[-(lower_order - 1):]) if lower_order > 1 else ()
        probability += weight * mle_prob(lower_order, lower_history, word)
    return max(probability, 1e-300)

# ------------------------------------------------------------
# 5. Good-Turing smoothing
# ------------------------------------------------------------
def good_turing_adjusted_count(r, frequency_of_frequencies):
    if r == 0:
        return frequency_of_frequencies.get(1, 0) / max(frequency_of_frequencies.get(0, 1), 1)
    next_frequency = frequency_of_frequencies.get(r + 1, 0)
    current_frequency = frequency_of_frequencies.get(r, 0)
    if next_frequency and current_frequency:
        return (r + 1) * next_frequency / current_frequency
    return float(r)

gt_cache = {}

def good_turing_distribution(order, history=()):
    history = tuple(history)
    cache_key = (order, history)
    if cache_key in gt_cache:
        return gt_cache[cache_key]

    if order == 1:
        observed = {word: counts[1].get((word,), 0) for word in vocab}
    else:
        observed = {
            word: counts[order].get(history + (word,), 0)
            for word in vocab
        }

    frequency_of_frequencies = Counter(count for count in observed.values() if count > 0)
    frequency_of_frequencies[0] = sum(count == 0 for count in observed.values())
    adjusted = {
        word: good_turing_adjusted_count(count, frequency_of_frequencies)
        for word, count in observed.items()
    }
    total = sum(adjusted.values())
    if total <= 0:
        result = {word: 1 / V for word in vocab}
    else:
        result = {word: value / total for word, value in adjusted.items()}
    gt_cache[cache_key] = result
    return result

def good_turing_prob(order, history, word):
    word = word_for_vocab(word)
    return good_turing_distribution(order, history).get(word, 0.0)

# ------------------------------------------------------------
# 6. Katz backoff smoothing
# ------------------------------------------------------------
# Katz uses Good-Turing discounting for seen n-grams and backs off to
# a lower-order model for unseen n-grams.
katz_alpha_cache = {}

def katz_prob(order, history, word):
    word = word_for_vocab(word)
    if order == 1:
        return good_turing_prob(1, (), word)

    history = tuple(word_for_vocab(token) for token in history)
    context_count = contexts[order].get(history, 0)
    if context_count == 0:
        return katz_prob(order - 1, history[1:], word)

    seen_words = followers[order].get(history, set())
    count = counts[order].get(history + (word,), 0)
    if count > 0:
        adjusted = good_turing_adjusted_count(
            count,
            Counter(value for value in counts[order].values() if value > 0)
        )
        return max((adjusted / count) * (count / context_count), 1e-300)

    cache_key = (order, history)
    if cache_key not in katz_alpha_cache:
        seen_mass = sum(
            katz_prob(order, history, seen_word)
            for seen_word in seen_words
        )
        lower_mass_for_unseen = 1.0 - sum(
            katz_prob(order - 1, history[1:], seen_word)
            for seen_word in seen_words
        )
        katz_alpha_cache[cache_key] = (
            max(1.0 - seen_mass, 1e-12) / max(lower_mass_for_unseen, 1e-12)
        )
    return katz_alpha_cache[cache_key] * katz_prob(order - 1, history[1:], word)

# ------------------------------------------------------------
# 7. Stupid backoff
# ------------------------------------------------------------
STUPID_BACKOFF_ALPHA = 0.4

def stupid_backoff_prob(order, history, word):
    word = word_for_vocab(word)
    if order == 1:
        return max(mle_prob(1, (), word), 1 / N)
    history = tuple(word_for_vocab(token) for token in history)
    if counts[order].get(history + (word,), 0) > 0:
        return max(mle_prob(order, history, word), 1e-300)
    return STUPID_BACKOFF_ALPHA * stupid_backoff_prob(order - 1, history[1:], word)

# ------------------------------------------------------------
# 8. Kneser-Ney smoothing
# ------------------------------------------------------------
KN_DISCOUNT = 0.75

def continuation_probability(word):
    word = word_for_vocab(word)
    total_unique_continuations = sum(len(values) for values in predecessors[2].values())
    return len(predecessors[2].get(word, set())) / max(total_unique_continuations, 1)

def kneser_ney_prob(order, history, word):
    word = word_for_vocab(word)
    if order == 1:
        return max(continuation_probability(word), 1e-300)

    history = tuple(word_for_vocab(token) for token in history)
    context_count = contexts[order].get(history, 0)
    if context_count == 0:
        return kneser_ney_prob(order - 1, history[1:], word)

    count = counts[order].get(history + (word,), 0)
    unique_followers = len(followers[order].get(history, set()))
    first_term = max(count - KN_DISCOUNT, 0.0) / context_count
    lambda_value = KN_DISCOUNT * unique_followers / context_count
    lower_probability = kneser_ney_prob(order - 1, history[1:], word)
    return max(first_term + lambda_value * lower_probability, 1e-300)

# ------------------------------------------------------------
# 9. Evaluate every requested technique on all four models
# ------------------------------------------------------------
methods = {
    "Interpolated": interpolated_prob,
    "Good-Turing": good_turing_prob,
    "Katz Backoff": katz_prob,
    "Stupid Backoff": stupid_backoff_prob,
    "Kneser-Ney": kneser_ney_prob,
}

def perplexity(sentences, probability_function, order):
    total_log_probability = 0.0
    total_tokens = 0
    for sentence in sentences:
        log_probability, token_count = sentence_log_probability(
            sentence, probability_function, order
        )
        total_log_probability += log_probability
        total_tokens += token_count
    return 2 ** (-total_log_probability / total_tokens)

results = {}
print("\nPerplexity results (lower is better)\n")
print(f"{'Method':<18} {'Model':<12} {'Dev PP':>15} {'Test PP':>15}")
print("-" * 64)
for method_name, probability_function in methods.items():
    for order in ORDERS:
        model_name = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "Quadrigram"}[order]
        dev_pp = perplexity(dev_sentences, probability_function, order)
        test_pp = perplexity(test_sentences, probability_function, order)
        results[(method_name, model_name)] = {"dev": dev_pp, "test": test_pp}
        print(f"{method_name:<18} {model_name:<12} {dev_pp:>15,.2f} {test_pp:>15,.2f}")

# ------------------------------------------------------------
# 10. Probability sanity checks and sample sentence comparison
# ------------------------------------------------------------
print("\nProbability sanity checks")
examples = [
    (2, (SOS,), "the"),
    (3, (SOS, SOS), "the"),
    (4, (SOS, SOS, SOS), "the"),
]
for order, history, word in examples:
    print(f"\nP({word} | {' '.join(history)})")
    for method_name, probability_function in methods.items():
        print(f"  {method_name:<18}: {probability_function(order, history, word):.10f}")

sample = test_sentences[0]
print("\nSample test sentence:")
print(" ".join(sample[:20]), "..." if len(sample) > 20 else "")
for method_name, probability_function in methods.items():
    print(f"\n{method_name}")
    for order in ORDERS:
        log_probability, token_count = sentence_log_probability(
            sample, probability_function, order
        )
        print(f"  {order}-gram log2 probability = {log_probability:.4f} ({token_count} predicted tokens)")

print("\nNotes:")
print("- Interpolation combines lower- and higher-order MLE probabilities.")
print("- Good-Turing reallocates probability mass using frequency-of-frequency counts.")
print("- Katz discounts seen n-grams and backs off when an n-gram is unseen.")
print("- Stupid backoff multiplies lower-order scores by 0.4; these are scores, not normalized probabilities.")
print("- Kneser-Ney uses absolute discounting and continuation probabilities.")

Corpus: c:\Users\kasha\OneDrive\Documents\ASH\5th\NLP\tokenized.txt
Sentences: train=998,000, dev=1,000, test=1,000
Vocabulary size: 592,492
1-gram types: 592,490; tokens: 13,274,039
2-gram types: 4,785,976; tokens: 13,274,039
3-gram types: 8,399,582; tokens: 13,274,039
4-gram types: 10,128,722; tokens: 13,274,039

Perplexity results (lower is better)

Method             Model                 Dev PP         Test PP
----------------------------------------------------------------
Interpolated       Unigram             4,085.36        4,284.56
Interpolated       Bigram       84,333,023,081.44 185,311,545,496.40
Interpolated       Trigram      77,449,206,682.05 174,621,216,546.73
Interpolated       Quadrigram   103,090,629,015.38 235,414,298,305.08
Good-Turing        Unigram             3,094.18        3,227.47
